In [ ]:
import pdfplumber
import pandas as pd
from datetime import datetime
import re

electric_details = {}
gas_details = {}

pdf_path = "BGE/20250716.pdf"
# pdf_path = "BGE/20250815.pdf"
# pdf_path = "BGE/20250616.pdf"

# Function returns standard date output; Jun22,2025 -> 2025-06-22
def format_date(input_date:str):
    # Expects abbreviated month name
    date_obj = datetime.strptime(input_date, "%b%d,%Y")
    format_date = date_obj.strftime("%Y-%m-%d")
    return format_date

def extract_electric(input_table:list, output_dict:dict):
    for item in input_table:
        row = item
        row = (item[0] or "").splitlines()
        # print(row)
        for tt in row:
            # Total Energy Used
            if "Current - Previous" in tt:
                output_dict["total_kWh"] = int(tt.split("= ")[1])
                break
            # Billing Periods
            if "BillingPeriod" in tt:
                date_pattern = r'[A-Z][a-z]{2}\d{1,2},\d{4}'
                dates = re.findall(date_pattern, tt)
                output_dict["billing_period_start"] = format_date(dates[0])
                output_dict["billing_period_end"] = format_date(dates[1])
                break

    # output_dict["electric_supply_0_rate"]
    # output_dict["electric_supply_1_rate"]
    # output_dict["electric_supply_2_rate"]

    # output_dict["total_kWh"] = input_table[3][1].splitlines()[0]

with pdfplumber.open(pdf_path) as pdf:
    first_page = pdf.pages[1]
    table = first_page.extract_tables()
    electric_table = table[0]
    gas_table = table[1]
    extract_electric(electric_table, electric_details)
    print(electric_details)
    # df = pd.DataFrame(table[1:], columns=table[0]) 



{'billing_period_start': '2025-05-22', 'billing_period_end': '2025-06-23', 'total_kWh': 751}
